In [450]:
import pandas as pd
import numpy as np
from nba_api.live.nba.endpoints import scoreboard
from nba_api.stats.endpoints import playergamelog, boxscoretraditionalv3, boxscoreusagev3, shotchartdetail, commonteamroster, playergamelogs
from typing import List, Dict, Optional, Union, Tuple
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import time

In [521]:
position_mapping = {'C': 0, 
                    'F': 1, 
                    'G': 2, 
                    '': 3}

min_game_date = datetime(2024, 7, 6)

team_mapping = {'MIN': 0,
 'PHX': 1,
 'SAC': 2,
 'CLE': 3,
 'TOR': 4,
 'DET': 5,
 'MEM': 6,
 'PHI': 7,
 'SAS': 8,
 'NOP': 9,
 'UTA': 10,
 'ORL': 11,
 'DEN': 12,
 'OKC': 13,
 'MIA': 14,
 'ATL': 15,
 'GSW': 16,
 'POR': 17,
 'HOU': 18,
 'DAL': 19,
 None: 20,
 'SKT': 21,
 'LAL': 22,
 'BOS': 23,
 'MIL': 24,
 'CHI': 25,
 'LAC': 26,
 'WAS': 27,
 'NYK': 28,
 'IND': 29,
 'CHA': 30,
 'BKN': 31}

In [522]:
def encode_position(position: str) -> int:
    try:
        return position_mapping[position]
    except ValueError:  # If unseen category
        print(f"Warning: Unrecognized position '{position}'")
        return -1  # Default unknown position encoding

In [523]:
encode_position('')

3

In [524]:
test_game = '0022400929'
test_player = '204001'
test_team = '1610612738'

In [525]:
scoreboard.ScoreBoard().games.get_dict()

[{'gameId': '0022400945',
  'gameCode': '20250312/CHAATL',
  'gameStatus': 2,
  'gameStatusText': 'Halftime            ',
  'period': 2,
  'gameClock': 'PT00M00.00S',
  'gameTimeUTC': '2025-03-12T23:30:00Z',
  'gameEt': '2025-03-12T19:30:00Z',
  'regulationPeriods': 4,
  'ifNecessary': False,
  'seriesGameNumber': '',
  'gameLabel': '',
  'gameSubLabel': '',
  'seriesText': '',
  'seriesConference': '',
  'poRoundDesc': '',
  'gameSubtype': '',
  'isNeutral': False,
  'homeTeam': {'teamId': 1610612737,
   'teamName': 'Hawks',
   'teamCity': 'Atlanta',
   'teamTricode': 'ATL',
   'wins': 31,
   'losses': 34,
   'score': 52,
   'seed': None,
   'inBonus': None,
   'timeoutsRemaining': 4,
   'periods': [{'period': 1, 'periodType': 'REGULAR', 'score': 30},
    {'period': 2, 'periodType': 'REGULAR', 'score': 22},
    {'period': 3, 'periodType': 'REGULAR', 'score': 0},
    {'period': 4, 'periodType': 'REGULAR', 'score': 0}]},
  'awayTeam': {'teamId': 1610612766,
   'teamName': 'Hornets',
   

In [526]:
def get_todays_games() -> pd.DataFrame:
    games = scoreboard.ScoreBoard().games.get_dict()
    game_list = []

    for game in games:
        game_list.append({
            "game_id": game["gameId"],
            "home_team": game["homeTeam"]["teamName"],
            "away_team": game["awayTeam"]["teamName"],
            "home_id": game["homeTeam"]["teamId"],
            "away_id": game["awayTeam"]["teamId"],
            "home_tri": game["homeTeam"]["teamTricode"],
            "away_tri": game["awayTeam"]["teamTricode"]

        })

    return pd.DataFrame(game_list)


In [527]:
get_todays_games().head(1)

,game_id,home_team,away_team,home_id,away_id,home_tri,away_tri
0,0022400946,Celtics,Thunder,1610612738,1610612760,BOS,OKC


In [528]:
def get_players_from_teams(team_ids: List[Tuple[int, int]]) -> List[Tuple[int,int]]:
    all_players = []

    for team_id, opp in team_ids:
        try:
            roster = commonteamroster.CommonTeamRoster(team_id=team_id)
            roster_df = roster.get_data_frames()[0]  # The roster table
            
            if not roster_df.empty:
                player_ids = roster_df["PLAYER_ID"].tolist()
                all_players.extend([(player_id, opp) for player_id in player_ids])
        
        except Exception as e:
            print(f"Error fetching roster for team {team_id}: {e}")

    return list(set(all_players)) 

In [529]:
get_players_from_teams([(1610612738, 2)])

[(1630573, 2),
 (1628436, 2),
 (1630202, 2),
 (204001, 2),
 (1627759, 2),
 (1628470, 2),
 (1641809, 2),
 (1628401, 2),
 (1630214, 2),
 (1628369, 2),
 (1629674, 2),
 (1641936, 2),
 (201143, 2),
 (201950, 2),
 (1641775, 2),
 (1631120, 2),
 (1631248, 2)]

In [530]:
def get_last_5_games(player_id: str, season:str = "2024-25") -> Tuple[pd.DataFrame, Optional[str], Optional[str]]:
    """Fetch last 5 games for a player."""
    try:
        logs = playergamelogs.PlayerGameLogs(season_nullable=season, player_id_nullable=player_id)
        df = logs.get_data_frames()[0]
        
        # Ensure we only take the last 5 games
        df = df.sort_values(by="GAME_DATE", ascending=False).head(5)
        last_game_id = df['GAME_ID'].iloc[0]
        last_game_date = df['GAME_DATE'].iloc[0]
        return df['GAME_ID'], last_game_id, last_game_date
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return None

In [531]:
ids, id, date = get_last_5_games('201143')
len(ids)

5

In [532]:
boxscore = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id='0022400928')
traditional_stats = boxscore.get_data_frames()[0]
traditional_stats[traditional_stats['personId'] == 1630249]['position'].tolist()

['']

In [533]:
traditional_stats.columns

Index(['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug',
       'personId', 'firstName', 'familyName', 'nameI', 'playerSlug',
       'position', 'comment', 'jerseyNum', 'minutes', 'fieldGoalsMade',
       'fieldGoalsAttempted', 'fieldGoalsPercentage', 'threePointersMade',
       'threePointersAttempted', 'threePointersPercentage', 'freeThrowsMade',
       'freeThrowsAttempted', 'freeThrowsPercentage', 'reboundsOffensive',
       'reboundsDefensive', 'reboundsTotal', 'assists', 'steals', 'blocks',
       'turnovers', 'foulsPersonal', 'points', 'plusMinusPoints'],
      dtype='object')

In [534]:
def m_to_s(min_str: str) -> int:
    try:
        minutes, seconds = map(int, min_str.split(":"))
        return minutes * 60 + seconds
    except:
        return 0

In [570]:
from nba_api.stats.endpoints import boxscoretraditionalv3, boxscoreusagev3, shotchartdetail

def get_game_stats(game_id: str, player_id:str) -> Optional[Dict[str, Union[float, int]]]:
    """Fetch traditional stats, usage stats, and shooting stats from the original endpoints."""
    stats = {}

    # traditional stats
    try:
        boxscore = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=str(game_id))
        traditional_stats = boxscore.get_data_frames()[0]
        player_stats = traditional_stats[traditional_stats["personId"] == int(player_id)]

        if not player_stats.empty:
            stats.update({
                "fieldGoalsMade": player_stats["fieldGoalsMade"].values[0],
                "fieldGoalsAttempted": player_stats["fieldGoalsAttempted"].values[0],
                "fieldGoalsPercentage": player_stats["fieldGoalsPercentage"].values[0],
                "threePointersMade": player_stats["threePointersMade"].values[0],
                "threePointersAttempted": player_stats["threePointersAttempted"].values[0],
                "threePointersPercentage": player_stats["threePointersPercentage"].values[0],
                "freeThrowsMade": player_stats["freeThrowsMade"].values[0],
                "freeThrowsAttempted": player_stats["freeThrowsAttempted"].values[0],
                "freeThrowsPercentage": player_stats["freeThrowsPercentage"].values[0],
                "reboundsOffensive": player_stats["reboundsOffensive"].values[0],
                "reboundsDefensive": player_stats["reboundsDefensive"].values[0],
                "reboundsTotal": player_stats["reboundsTotal"].values[0],
                "assists": player_stats["assists"].values[0],
                "steals": player_stats["steals"].values[0],
                "blocks": player_stats["blocks"].values[0],
                "turnovers": player_stats["turnovers"].values[0],
                "foulsPersonal": player_stats["foulsPersonal"].values[0],
                "points": player_stats["points"].values[0],
                "plusMinusPoints": player_stats["plusMinusPoints"].values[0],
                "MIN": m_to_s(player_stats["minutes"].values[0]),
                "position_encoded": encode_position(str(player_stats["position"].values[0])),
            })
    except Exception as e:
        print(f"Error fetching traditional stats for {player_id} in game {game_id}: {e}")

    # usage stats
    try:
        usage = boxscoreusagev3.BoxScoreUsageV3(game_id=game_id)
        usage_stats = usage.get_data_frames()[0]
        player_usage = usage_stats[usage_stats["personId"] == str(player_id)]

        if not player_usage.empty:
            stats["usagePercentage"] = player_usage["USG_PCT"].values[0]
    except Exception as e:
        print(f"Error fetching usage stats for {player_id} in game {game_id}: {e}")

    # shot chart data
    try:
        shot_data = shotchartdetail.ShotChartDetail(team_id=0, player_id=str(player_id), season_nullable="2024-25", context_measure_simple="FGA")
        shots = shot_data.get_data_frames()[0]
        
        if not shots.empty:
            stats.update({
                "total_shots": shots["SHOT_ATTEMPTED_FLAG"].sum(),
                "total_makes": shots["SHOT_MADE_FLAG"].sum(),
                "fg_pct": shots["SHOT_MADE_FLAG"].sum() / max(1, shots["SHOT_ATTEMPTED_FLAG"].sum()),
                "paint_shots": (shots["SHOT_ZONE_BASIC"] == "Restricted Area").sum(),
                "mid_range_shots": (shots["SHOT_ZONE_BASIC"] == "Mid-Range").sum(),
                "three_point_shots": (shots["SHOT_ZONE_BASIC"] == "Above the Break 3").sum(),
                "avg_distance": shots["SHOT_DISTANCE"].mean(),
            })
    except Exception as e:
        print(f"Error fetching shot chart stats for {player_id} in game {game_id}: {e}")

    return stats


In [571]:
get_game_stats('0022400918', '201143')

{'fieldGoalsMade': 5,
 'fieldGoalsAttempted': 9,
 'fieldGoalsPercentage': 0.556,
 'threePointersMade': 1,
 'threePointersAttempted': 3,
 'threePointersPercentage': 0.333,
 'freeThrowsMade': 3,
 'freeThrowsAttempted': 4,
 'freeThrowsPercentage': 0.75,
 'reboundsOffensive': 1,
 'reboundsDefensive': 8,
 'reboundsTotal': 9,
 'assists': 4,
 'steals': 3,
 'blocks': 1,
 'turnovers': 0,
 'foulsPersonal': 1,
 'points': 14,
 'plusMinusPoints': 16.0,
 'MIN': 2239,
 'position_encoded': 0,
 'total_shots': 361,
 'total_makes': 151,
 'fg_pct': 0.4182825484764543,
 'paint_shots': 62,
 'mid_range_shots': 9,
 'three_point_shots': 158,
 'avg_distance': 18.42105263157895}

In [572]:
def compute_rolling_stats(player_id: str) -> Optional[Dict[str, Union[float, int]]]:
    """Fetch last 5 games for a player, calculate rolling stats, and return as a dictionary."""
    try:
        # Get the last 5 games for the player
        last_5_games, last_game, date = get_last_5_games(player_id)

        # If player has less than 2 past games, return None (not enough data)
        if len(last_5_games) < 2:
            print(f"Not enough games for rolling stats for player {player_id}")
            return None

        # Fetch stats for each of those games
        game_stats_list = [get_game_stats(game_id, player_id) for game_id in last_5_games]

        # Remove any None values in case of failed API calls
        game_stats_list = [stats for stats in game_stats_list if stats]

        if len(game_stats_list) < 2:
            print(f"Not enough valid game stats for rolling stats for player {player_id}")
            return None

        # Convert list of dictionaries into DataFrame
        df = pd.DataFrame(game_stats_list)

        # Compute rolling averages
        rolling_stats = df.mean().to_dict()

        # Rename keys to indicate they are rolling averages
        rolling_stats = {f"prev_ROLLING_{key}": value for key, value in rolling_stats.items()}

        return rolling_stats

    except Exception as e:
        print(f"Error computing rolling stats for player {player_id}: {e}")
        return None

In [573]:
compute_rolling_stats('201143')

{'prev_ROLLING_fieldGoalsMade': 4.2,
 'prev_ROLLING_fieldGoalsAttempted': 8.8,
 'prev_ROLLING_fieldGoalsPercentage': 0.4766,
 'prev_ROLLING_threePointersMade': 1.6,
 'prev_ROLLING_threePointersAttempted': 4.6,
 'prev_ROLLING_threePointersPercentage': 0.457,
 'prev_ROLLING_freeThrowsMade': 1.8,
 'prev_ROLLING_freeThrowsAttempted': 2.0,
 'prev_ROLLING_freeThrowsPercentage': 0.55,
 'prev_ROLLING_reboundsOffensive': 1.4,
 'prev_ROLLING_reboundsDefensive': 7.2,
 'prev_ROLLING_reboundsTotal': 8.6,
 'prev_ROLLING_assists': 2.4,
 'prev_ROLLING_steals': 1.6,
 'prev_ROLLING_blocks': 0.8,
 'prev_ROLLING_turnovers': 1.0,
 'prev_ROLLING_foulsPersonal': 1.2,
 'prev_ROLLING_points': 11.8,
 'prev_ROLLING_plusMinusPoints': 4.4,
 'prev_ROLLING_MIN': 2055.4,
 'prev_ROLLING_position_encoded': 0.6,
 'prev_ROLLING_total_shots': 361.0,
 'prev_ROLLING_total_makes': 151.0,
 'prev_ROLLING_fg_pct': 0.41828254847645424,
 'prev_ROLLING_paint_shots': 62.0,
 'prev_ROLLING_mid_range_shots': 9.0,
 'prev_ROLLING_three_

In [574]:
def get_clutch_stats(player_id: str, season: str = "2024-25") -> Optional[Dict[str, Union[float, int]]]:
    try:
        clutch_shots = shotchartdetail.ShotChartDetail(
            team_id=0, 
            player_id=player_id, 
            season_nullable=season, 
            context_measure_simple="FGA", 
            clutch_time_nullable="Last 5 Minutes"
        )
        
        clutch_df = clutch_shots.get_data_frames()[0]

        if clutch_df.empty:
            return {
                "total_shots_clutch": 0, "total_makes_clutch": 0, "fg_pct_clutch": 0,
                "paint_shots_clutch": 0, "mid_range_shots_clutch": 0,
                "three_point_shots_clutch": 0, "avg_distance_clutch": 0
            }

        # Aggregate clutch stats
        clutch_stats = {
            "total_shots_clutch": clutch_df["SHOT_ATTEMPTED_FLAG"].sum(),
            "total_makes_clutch": clutch_df["SHOT_MADE_FLAG"].sum(),
            "fg_pct_clutch": clutch_df["SHOT_MADE_FLAG"].sum() / max(1, clutch_df["SHOT_ATTEMPTED_FLAG"].sum()),
            "paint_shots_clutch": (clutch_df["SHOT_ZONE_BASIC"] == "Restricted Area").sum(),
            "mid_range_shots_clutch": (clutch_df["SHOT_ZONE_BASIC"] == "Mid-Range").sum(),
            "three_point_shots_clutch": (clutch_df["SHOT_ZONE_BASIC"] == "Above the Break 3").sum(),
            "avg_distance_clutch": clutch_df["SHOT_DISTANCE"].mean(),
        }

        return clutch_stats

    except Exception as e:
        print(f"Error fetching clutch stats for player {player_id}: {e}")
        return {
            "total_shots_clutch": 0, "total_makes_clutch": 0, "fg_pct_clutch": 0,
            "paint_shots_clutch": 0, "mid_range_shots_clutch": 0,
            "three_point_shots_clutch": 0, "avg_distance_clutch": 0
        }

In [575]:
get_clutch_stats('201143')

{'total_shots_clutch': 34,
 'total_makes_clutch': 11,
 'fg_pct_clutch': 0.3235294117647059,
 'paint_shots_clutch': 6,
 'mid_range_shots_clutch': 1,
 'three_point_shots_clutch': 13,
 'avg_distance_clutch': 19.41176470588235}

In [576]:
todays_games = get_todays_games()
[(row["home_id"], team_mapping[row["away_tri"]])
    for _, row in todays_games.iterrows()
] + [
    (row["away_id"], team_mapping[row["home_tri"]])
    for _, row in todays_games.iterrows()
]

[(1610612737, 30),
 (1610612738, 13),
 (1610612761, 7),
 (1610612748, 26),
 (1610612745, 1),
 (1610612763, 10),
 (1610612759, 19),
 (1610612743, 0),
 (1610612757, 28),
 (1610612766, 15),
 (1610612760, 23),
 (1610612755, 4),
 (1610612746, 14),
 (1610612756, 18),
 (1610612762, 6),
 (1610612742, 8),
 (1610612750, 12),
 (1610612752, 17)]

In [577]:
def prepare_model_input() -> pd.DataFrame:
    """
    Fetch all required stats and prepare them in the correct order for model input.

    Returns:
        pd.DataFrame: A DataFrame where each row corresponds to a player's input features.
    """
    # Step 1: Get today's games and players
    todays_games = get_todays_games()
    all_team_ids = [(row["home_id"], team_mapping[row["away_tri"]])
            for _, row in todays_games.iterrows()
        ] + [
            (row["away_id"], team_mapping[row["home_tri"]])
            for _, row in todays_games.iterrows()
        ]
    players_today = get_players_from_teams(all_team_ids)

    # Step 2: Initialize list to store player data
    final_data = []

    for player_id, opp in players_today:
        try:
            # Step 3: Get last game ID and stats
            # print(f'before getting games: {player_id} and {opp}')
            last_5_games, last_game_id, date = get_last_5_games(player_id)
            # print('middle of getting games')
            if  len(last_5_games) == 0:
                continue  # Skip if no previous games found
            
            # print('past getting games')
            prev_game_stats = get_game_stats(last_game_id, player_id)

            # Step 4: Compute rolling stats
            rolling_stats = compute_rolling_stats(player_id)

            # Step 5: Calculate TS% and eFG%
            fga = prev_game_stats.get("FGA", 0)
            fta = prev_game_stats.get("FTA", 0)
            pts = prev_game_stats.get("PTS", 0)
            fgm = prev_game_stats.get("FGM", 0)
            fg3m = prev_game_stats.get("FG3M", 0)

            prev_game_stats["ts"] = pts / (2 * (fga + (0.44 * fta))) if (fga + (0.44 * fta)) > 0 else 0
            prev_game_stats["efg"] = (fgm + (0.5 * fg3m)) / fga if fga > 0 else 0

            # Step 6: Fetch clutch stats
            clutch_stats = get_clutch_stats(player_id)

            # Step 7: Combine all stats into a single record
            full_stats = {**prev_game_stats, **rolling_stats, **clutch_stats}
            
            # Ensure all features are correctly prefixed
            formatted_stats = {f"prev_{k}" if not k.startswith("prev_") else k: v for k, v in full_stats.items()}

            # Step 8: Add position encoding, days since last game, and opposing team
            formatted_stats["prev_days_since"] = (pd.to_datetime(date) - min_game_date).days
            formatted_stats["prev_opposing_team_numeric"] = opp

            final_data.append(formatted_stats)

        except Exception as e:
            print(f"Error processing player {player_id}: {e}")

    # Step 9: Convert to DataFrame and order columns
    df = pd.DataFrame(final_data)

    # Ensure correct feature order (same as before)
    feature_order = [
        "prev_minutes", "prev_fieldGoalsMade", "prev_fieldGoalsAttempted", "prev_fieldGoalsPercentage",
        "prev_threePointersMade", "prev_threePointersAttempted", "prev_threePointersPercentage",
        "prev_freeThrowsMade", "prev_freeThrowsAttempted", "prev_freeThrowsPercentage",
        "prev_reboundsOffensive", "prev_reboundsDefensive", "prev_reboundsTotal", "prev_assists",
        "prev_steals", "prev_blocks", "prev_turnovers", "prev_foulsPersonal", "prev_points",
        "prev_plusMinusPoints", "prev_WL", "prev_MIN", "prev_PTS", "prev_FGM", "prev_FGA",
        "prev_FG_PCT", "prev_FG3M", "prev_FG3A", "prev_FG3_PCT", "prev_FTM", "prev_FTA",
        "prev_FT_PCT", "prev_OREB", "prev_DREB", "prev_REB", "prev_AST", "prev_STL", "prev_BLK",
        "prev_TOV", "prev_PF", "prev_PLUS_MINUS", "prev_total_shots", "prev_total_makes", "prev_fg_pct",
        "prev_paint_shots", "prev_mid_range_shots", "prev_three_point_shots", "prev_avg_distance",
        "prev_total_shots_clutch", "prev_total_makes_clutch", "prev_fg_pct_clutch",
        "prev_paint_shots_clutch", "prev_mid_range_shots_clutch", "prev_three_point_shots_clutch",
        "prev_avg_distance_clutch", "prev_efg", "prev_ts", "prev_streak", "prev_usagePercentage",
        "prev_ROLLING_fieldGoalsMade", "prev_ROLLING_fieldGoalsAttempted", "prev_ROLLING_fieldGoalsPercentage",
        "prev_ROLLING_threePointersMade", "prev_ROLLING_threePointersAttempted", "prev_ROLLING_threePointersPercentage",
        "prev_ROLLING_freeThrowsMade", "prev_ROLLING_freeThrowsAttempted", "prev_ROLLING_freeThrowsPercentage",
        "prev_ROLLING_reboundsOffensive", "prev_ROLLING_reboundsDefensive", "prev_ROLLING_reboundsTotal",
        "prev_ROLLING_assists", "prev_ROLLING_steals", "prev_ROLLING_blocks", "prev_ROLLING_turnovers",
        "prev_ROLLING_foulsPersonal", "prev_ROLLING_points", "prev_ROLLING_plusMinusPoints", "prev_ROLLING_MIN",
        "prev_ROLLING_PTS", "prev_ROLLING_FGM", "prev_ROLLING_FGA", "prev_ROLLING_FG_PCT", "prev_ROLLING_FG3M",
        "prev_ROLLING_FG3A", "prev_ROLLING_FG3_PCT", "prev_ROLLING_FTM", "prev_ROLLING_FTA", "prev_ROLLING_FT_PCT",
        "prev_ROLLING_OREB", "prev_ROLLING_DREB", "prev_ROLLING_REB", "prev_ROLLING_AST", "prev_ROLLING_STL",
        "prev_ROLLING_BLK", "prev_ROLLING_TOV", "prev_ROLLING_PF", "prev_ROLLING_PLUS_MINUS",
        "prev_position_encoded", "prev_days_since", "prev_opposing_team_numeric"
    ]
    df = df.reindex(columns=feature_order, fill_value=0)

    return df

In [578]:
prepare_model_input()

Not enough games for rolling stats for player 1642434
Error processing player 1642434: 'NoneType' object is not a mapping
Error fetching game logs for player 1641747: single positional indexer is out-of-bounds
Error processing player 1641747: cannot unpack non-iterable NoneType object
Not enough games for rolling stats for player 1641817
Error processing player 1641817: 'NoneType' object is not a mapping
Error fetching game logs for player 1630585: single positional indexer is out-of-bounds
Error processing player 1630585: cannot unpack non-iterable NoneType object
Error fetching usage stats for 1631117 in game 0022400843: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching clutch stats for player 1631117: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching shot chart stats for 1641711 in game 0022400870: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=

,prev_minutes,prev_fieldGoalsMade,prev_fieldGoalsAttempted,prev_fieldGoalsPercentage,prev_threePointersMade,prev_threePointersAttempted,prev_threePointersPercentage,prev_freeThrowsMade,prev_freeThrowsAttempted,prev_freeThrowsPercentage,...,prev_ROLLING_REB,prev_ROLLING_AST,prev_ROLLING_STL,prev_ROLLING_BLK,prev_ROLLING_TOV,prev_ROLLING_PF,prev_ROLLING_PLUS_MINUS,prev_position_encoded,prev_days_since,prev_opposing_team_numeric
0,0,3,6,0.500,2,5,0.400,0,0,0.000,...,0,0,0,0,0,0,0,3,247,30
1,0,4,5,0.800,0,0,0.000,2,6,0.333,...,0,0,0,0,0,0,0,3,247,18
2,0,2,8,0.250,0,4,0.000,0,1,0.000,...,0,0,0,0,0,0,0,1,238,8
3,0,1,9,0.111,0,5,0.000,0,0,0.000,...,0,0,0,0,0,0,0,1,247,10
4,0,0,1,0.000,0,1,0.000,0,0,0.000,...,0,0,0,0,0,0,0,3,246,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,0,5,8,0.625,2,3,0.667,0,0,0.000,...,0,0,0,0,0,0,0,3,247,23
303,0,1,1,1.000,0,0,0.000,1,2,0.500,...,0,0,0,0,0,0,0,0,247,19
304,0,1,2,0.500,0,1,0.000,0,0,0.000,...,0,0,0,0,0,0,0,3,247,17
305,0,1,2,0.500,1,1,1.000,0,0,0.000,...,0,0,0,0,0,0,0,3,246,12


In [582]:
shot_data = shotchartdetail.ShotChartDetail(team_id=0, player_id='1630249', season_nullable="2024-25", context_measure_simple="FGA")
shot_data.get_data_frames()[0]

,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM
0,Shot Chart Detail,0022400030,515,1630249,Vít Krejčí,1610612737,Atlanta Hawks,4,11,39,...,Center(C),Less Than 8 ft.,2,7,19,1,0,20241122,CHI,ATL
1,Shot Chart Detail,0022400030,531,1630249,Vít Krejčí,1610612737,Atlanta Hawks,4,10,56,...,Center(C),Less Than 8 ft.,3,6,38,1,0,20241122,CHI,ATL
2,Shot Chart Detail,0022400064,238,1630249,Vít Krejčí,1610612737,Atlanta Hawks,2,6,23,...,Right Side Center(RC),24+ ft.,25,165,200,1,0,20241023,ATL,BKN
3,Shot Chart Detail,0022400064,583,1630249,Vít Krejčí,1610612737,Atlanta Hawks,4,8,21,...,Center(C),Less Than 8 ft.,1,-12,-2,1,1,20241023,ATL,BKN
4,Shot Chart Detail,0022400079,109,1630249,Vít Krejčí,1610612737,Atlanta Hawks,1,4,49,...,Left Side(L),24+ ft.,24,-235,71,1,0,20241025,ATL,CHA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,Shot Chart Detail,0022400945,256,1630249,Vít Krejčí,1610612737,Atlanta Hawks,2,5,52,...,Center(C),24+ ft.,26,55,263,1,1,20250312,ATL,CHA
221,Shot Chart Detail,0022400945,573,1630249,Vít Krejčí,1610612737,Atlanta Hawks,4,9,9,...,Center(C),24+ ft.,27,-72,261,1,1,20250312,ATL,CHA
222,Shot Chart Detail,0022400945,599,1630249,Vít Krejčí,1610612737,Atlanta Hawks,4,7,49,...,Right Side(R),8-16 ft.,12,121,27,1,0,20250312,ATL,CHA
223,Shot Chart Detail,0022401202,520,1630249,Vít Krejčí,1610612737,Atlanta Hawks,4,10,57,...,Center(C),Less Than 8 ft.,2,-1,23,1,0,20241211,NYK,ATL


In [579]:
todays_games = get_todays_games()
all_team_ids = [(row["home_id"], team_mapping[row["away_tri"]])
        for _, row in todays_games.iterrows()
    ] + [
        (row["away_id"], team_mapping[row["home_tri"]])
        for _, row in todays_games.iterrows()
    ]
players_today = get_players_from_teams(all_team_ids)
players_today

[(1630249, 30),
 (203486, 18),
 (1629023, 8),
 (1642377, 10),
 (1642461, 0),
 (1627750, 0),
 (203994, 15),
 (1631124, 0),
 (1642434, 19),
 (1629646, 19),
 (1628381, 6),
 (1631096, 23),
 (1629599, 14),
 (1630811, 30),
 (1642530, 10),
 (1631127, 19),
 (1641747, 0),
 (1642271, 6),
 (1627832, 1),
 (1630625, 28),
 (1631223, 1),
 (1642419, 7),
 (1629626, 18),
 (1641739, 28),
 (1630215, 4),
 (201935, 14),
 (1628991, 10),
 (1631116, 14),
 (1631210, 30),
 (1630552, 30),
 (1629098, 1),
 (1629618, 0),
 (203952, 26),
 (203083, 4),
 (1642278, 17),
 (1641765, 8),
 (1629656, 4),
 (1631217, 15),
 (201145, 1),
 (1631111, 15),
 (1641803, 12),
 (1626157, 17),
 (1629234, 14),
 (1628983, 23),
 (1642258, 30),
 (1629675, 12),
 (1630556, 8),
 (1630168, 30),
 (1642505, 23),
 (1642263, 1),
 (1642285, 10),
 (1630534, 7),
 (203944, 12),
 (1631115, 7),
 (1641817, 17),
 (1641779, 18),
 (1631243, 30),
 (201143, 13),
 (203078, 18),
 (1631107, 26),
 (1630545, 12),
 (1626220, 18),
 (201599, 0),
 (1631121, 28),
 (203967